In [2]:
import pandas as pd

In [3]:
DATA_DIR = "../../data/"
OUTPUT_DIR = "../../outputs/"

# Carregar dados dos parlamentares

In [4]:
df_parls_raw = pd.read_csv(
    f"{DATA_DIR}/df_parlamentares_por_legislatura.csv", 
)

legislaturas = df_parls_raw["idLegislatura"].unique()

# parlamentares únicos por legislatura
df_parls_raw = df_parls_raw[['id', 'idLegislatura']].drop_duplicates()

df_parl_detalhes = pd.read_csv(
    f"{DATA_DIR}/df_parl_detalhes.csv", index_col=0, dtype={"cpf": str}
)

df_parls_raw = df_parls_raw.merge(df_parl_detalhes, left_on="id", right_on="id", how="left")

df_parls_raw["cpf"] = df_parls_raw["cpf"].astype(str).str.zfill(11)
df_parls_raw.groupby("idLegislatura").size()

idLegislatura
48    589
49    620
50    635
51    642
52    626
53    636
54    671
55    623
56    613
57    595
dtype: int64

## Load race data

In [5]:
df_tse_candidatos = pd.read_csv(f"{DATA_DIR}/df_tse_candidatos.csv")
df_tse_candidatos["cpf"] = df_tse_candidatos["cpf"].astype(str).str.zfill(11)

df_tse_candidatos = df_tse_candidatos[["cpf", "raca"]].dropna()



In [6]:
df_parls_raw_with_race = df_parls_raw.merge(
    df_tse_candidatos.drop_duplicates(subset="cpf", keep="last"), on="cpf", how="left" # TODO, estou mantendo apenas o último registro...
)
df_parls_raw_with_race.groupby("idLegislatura").size()

idLegislatura
48    589
49    620
50    635
51    642
52    626
53    636
54    671
55    623
56    613
57    595
dtype: int64

## Add race data

In [9]:
# Equality data
df_parls = df_parls_raw.copy()

df_parls = df_parls.join(pd.get_dummies(df_parls["sexo"]))
df_parls = df_parls.join(pd.get_dummies(df_parls["escolaridade"]))
df_parls = df_parls.join(pd.get_dummies(df_parls_raw_with_race["raca"]))

df_parls.set_index(["id", "idLegislatura"], inplace=True)


## Define your population proportions
population_proportions = {
    "F": 0.52,
    "M": 0.48,
    "branca": 0.2,
    "n_branca": 0.8,
    "parda": .3,
    "preta": .3,
    "amarela": .3,
    "indigena": .3,
}

df_parls["n_branca"] = (
    df_parls["parda"]
    + df_parls["preta"]
    + df_parls["amarela"]
    + df_parls["indigena"]
)
df_for_equality = df_parls[population_proportions.keys()].astype(int).copy()


In [11]:
df_for_equality.loc[pd.IndexSlice[:, 56], :].describe()

,F,M,branca,n_branca,parda,preta,amarela,indigena
count,613.000000,613.000000,613.000000,613.000000,613.000000,613.000000,613.000000,613.000000
mean,0.148450,0.851550,0.753670,0.246330,0.197390,0.044046,0.003263,0.001631
std,0.355836,0.355836,0.431225,0.431225,0.398354,0.205364,0.057073,0.040390
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


<!-- ## Load IPP data -->

## Salvar

In [12]:
df_for_equality.to_csv(f"{OUTPUT_DIR}/df_for_ird.csv")